In [ ]:
!pip install nest_asyncio


In [ ]:
import asyncio
import pandas as pd
import nest_asyncio

# Apply nest_asyncio to allow running asynchronous code in environments with active event loops
nest_asyncio.apply()

async def read_transactions():
    # Simulate reading transactions from the dataset
    df = pd.read_csv("/content/basic data (fraud detection)proto 1.csv")
    for _, transaction in df.iterrows():  # Synchronous iteration, no need for async/await here
        yield transaction
    await asyncio.sleep(1)  # Simulate a pause between readings

async def calculate_frequency(transactions):
    transaction_counts_per_day = pd.Series(dtype=int)
    thresholds = pd.Series(dtype=float)

    async for transaction in transactions:
        transaction_date = pd.to_datetime(transaction['Date'], dayfirst=True)  # Specify dayfirst to avoid parsing issues

        # Calculate transaction frequency per day
        transaction_counts_per_day[transaction_date] = transaction_counts_per_day.get(transaction_date, 0) + 1
        # Keep only the last 5 days of data
        transaction_counts_per_day = transaction_counts_per_day[-5:]

        # Calculate threshold and detect anomalies
        if len(transaction_counts_per_day) >= 5:
            threshold = transaction_counts_per_day.mean() + 2 * transaction_counts_per_day.std()

            # Adjust threshold if significant change
            if len(thresholds) > 1:
                prev_threshold = thresholds.iloc[-2]
                if abs(threshold - prev_threshold) > 0.1 * prev_threshold:
                    threshold = prev_threshold * 0.9  # Adjust by 10%

            thresholds[transaction_date] = threshold

            if transaction_counts_per_day.iloc[-1] > threshold:
                print(f"Anomalous transaction frequency detected on {transaction_date}: {transaction_counts_per_day.iloc[-1]} transactions")
                print(f"Threshold adjusted to: {threshold}")

async def main():
    transactions = read_transactions()
    await calculate_frequency(transactions)  # Await the coroutine to avoid "never awaited" warnings

# Use an event loop to run the coroutine
await main()  # Awaiting directly in a running event loop


In [ ]:
import asyncio
import pandas as pd
import nest_asyncio

# Apply nest_asyncio to allow running asynchronous code in environments with active event loops
nest_asyncio.apply()

async def read_transactions():
    # Simulate reading transactions from the dataset
    df = pd.read_csv("/content/basic data (fraud detection)proto 1.csv")  # Change this to the correct path
    for _, transaction in df.iterrows():  # Synchronous iteration, no need for async/await here
        yield transaction
    await asyncio.sleep(1)  # Simulate a pause between readings

async def calculate_frequency(transactions):
    transaction_counts_per_day = pd.Series(dtype=int)
    thresholds = pd.Series(dtype=float)

    async for transaction in transactions:
        transaction_date = pd.to_datetime(transaction['Date'], dayfirst=True)  # Specify dayfirst to avoid parsing issues

        # Calculate transaction frequency per day
        transaction_counts_per_day[transaction_date] = transaction_counts_per_day.get(transaction_date, 0) + 1
        # Keep only the last 5 days of data
        transaction_counts_per_day = transaction_counts_per_day[-5:]

        # Calculate threshold and detect anomalies
        if len(transaction_counts_per_day) >= 5:
            mean = transaction_counts_per_day.mean()
            std_dev = transaction_counts_per_day.std()
            threshold = mean + 2 * std_dev

            # Adjust threshold if significant change
            if len(thresholds) > 1:
                prev_threshold = thresholds.iloc[-1]
                if abs(threshold - prev_threshold) > 0.1 * prev_threshold:
                    threshold = prev_threshold * 0.9  # Adjust by 10%

            thresholds[transaction_date] = threshold

            # Check if the most recent transaction count exceeds the threshold
            latest_transaction_count = transaction_counts_per_day.iloc[-1]
            if latest_transaction_count > threshold:
                print(f"Anomalous transaction frequency detected on {transaction_date}: {latest_transaction_count} transactions")
                print(f"Threshold adjusted to: {threshold}")

async def main():
    transactions = read_transactions()  # Get the transactions asynchronously
    await calculate_frequency(transactions)  # Await the coroutine to avoid "never awaited" warnings

# Run the coroutine in environments with active event loops
await main()  # Awaiting directly in a running event loop


In [ ]:
import asyncio
import pandas as pd
import nest_asyncio


nest_asyncio.apply()  # Allows running asynchronous code in environments with active event loops

# Path to the CSV file
csv_path = "/content/basic data (fraud detection)proto 1.csv"

# Read the CSV data
df = pd.read_csv(csv_path)

# Display the first few rows to verify data reading
print(df.head())  # Check if this prints valid data
# Apply nest_asyncio to allow running asynchronous code in environments with active event loops
nest_asyncio.apply()

async def read_transactions():
    df = pd.read_csv("/content/basic data (fraud detection)proto 1.csv")
    transactions_per_day = df.groupby("Date").size()  # Group by date and count transactions
    return transactions_per_day

async def calculate_thresholds(transactions_per_day):
    thresholds = pd.Series(dtype=float)

    # Use a consistent window of the last 5 days to calculate the threshold
    for date, count in transactions_per_day.items():  # Changed 'iteritems' to 'items'
        if len(thresholds) >= 5:
            last_5_counts = transactions_per_day[-5:]
            mean = last_5_counts.mean()
            std_dev = last_5_counts.std()
            threshold = mean + 2 * std_dev

            # Threshold adjustment logic (only if significant change)
            if len(thresholds) > 1:
                prev_threshold = thresholds.iloc[-1]
                if abs(threshold - prev_threshold) > 0.1 * prev_threshold:
                    threshold = prev_threshold * 0.9  # Adjust by 10%

            thresholds[date] = threshold

            # Detect anomaly if count exceeds threshold
            if count > threshold:
                print(f"Anomalous transaction frequency detected on {date}: {count} transactions")
                print(f"Threshold adjusted to: {threshold}")

async def main():
    transactions_per_day = await read_transactions()
    await calculate_thresholds(transactions_per_day)  # Await the coroutine to avoid "never awaited" warnings

# Use an event loop to run the coroutine
await main()  # Awaiting directly in a running event loop


In [ ]:
import asyncio
import pandas as pd
import nest_asyncio

# Step 1: Apply nest_asyncio to ensure async compatibility with running event loops
nest_asyncio.apply()

# Step 2: Define the CSV path and read the data
csv_path = "/content/basic data (fraud detection)proto 1.csv"

# Try reading the data
try:
    df = pd.read_csv(csv_path)  # Read the CSV data
    print("Data read successfully.")

    # Display the first few rows of the data to ensure it's correct
    print("CSV Data Sample:")
    print(df.head())
except Exception as e:
    print(f"Error reading CSV data: {e}")
    raise  # If there's an error, re-raise it to halt execution

# Step 3: Debug Data Processing
# Check if the Date column is correctly parsed
try:
    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)  # Parse dates with the dayfirst option
    print("Date parsing successful.")
except Exception as e:
    print(f"Error parsing dates: {e}")
    raise  # If there's an error, re-raise it to halt execution

# Group by "Date" and count the number of transactions per day
transactions_per_day = df.groupby("Date").size()

# Step 4: Check Data Aggregation
# Print the transaction counts per day to verify the aggregation
print("Transaction counts per day:")
print(transactions_per_day)

# Asynchronous function to calculate thresholds and detect anomalies
async def calculate_thresholds(transactions_per_day):
    thresholds = pd.Series(dtype=float)  # Create an empty series to store thresholds

    # Loop over the transaction counts by day
    for date, count in transactions_per_day.items():
        print(f"Processing data for {date}: {count} transactions.")  # Debug print

        # Step 5: Check Threshold Calculation and Anomaly Detection
        # If there are at least 5 data points, calculate the threshold
        if len(thresholds) >= 5:
            last_5_counts = transactions_per_day.tail(5)  # Last 5 transaction counts
            mean = last_5_counts.mean()  # Mean of the last 5 days
            std_dev = last_5_counts.std()  # Standard deviation of the last 5 days
            threshold = mean + 2 * std_dev  # Calculate threshold

            # Adjust the threshold if needed based on recent changes
            if len(thresholds) > 1:
                prev_threshold = thresholds.iloc[-2]
                if abs(threshold - prev_threshold) > 0.1 * prev_threshold:
                    threshold = prev_threshold * 0.9  # Adjust by 10% to avoid drastic changes

            thresholds[date] = threshold  # Store the threshold for this date

            # Anomaly detection: If count exceeds the threshold, it's anomalous
            if count > threshold:
                print(f"Anomalous transaction frequency detected on {date}: {count} transactions.")
                print(f"Threshold adjusted to: {threshold}")
        else:
            print(f"Not enough data for threshold calculation. Current data count: {len(thresholds)}")

# Asynchronous main function to check for anomalies
async def main():
    await calculate_thresholds(transactions_per_day)  # Await the coroutine to ensure proper execution

# Use an event loop to run the asynchronous main function
await main()  # Awaiting directly in a running event loop
